# 💬 شام — مرحلة المحادثة (المرحلة الثالثة)

تُكمل من ناتج المرحلة الثانية (`sham-multimodal-checkpoint`) أو من ناتجها السابق (`sham-chat-checkpoint`)، وتعلّم **نفس دماغ شام** أن **يجيب**:

1. **صيغة حوار شام**: `<BOS> <USER> سؤال <SHAM> جواب <EOS>` — الخسارة على الجواب فقط، ويتعلّم متى **يتوقف**.
2. **وسائط داخل المحادثة**: «صف هذه الصورة» ← وصف، «ارسم: …» ← صورة، «اكتب ما يقال» ← نص، «انطق: …» ← صوت. الرد نفسه قد يكون صورة أو صوتاً من نفس النموذج.
3. **التذكّر**: نص عربي من ويكيبيديا وأزواج صورة/صوت ممزوجة بالحوار، حتى لا ينسى ما تعلّمه.
4. **المقياس الثابت**: قبل التدريب وبعده، على أسئلة عربية لم يُدرَّب عليها أبداً، ويُرسل التقرير إلى تيليجرام.

البيانات: `arbml/CIDAR` و`CohereLabs/aya_dataset` (العربية)، كلاهما apache-2.0 ومكتوب بأيدٍ بشرية. كل تشغيل يأخذ أمثلة **جديدة** ويكمل بعد آخر موضع.

**الإعداد:** Accelerator: أي خيار (GPU أو None). Internet: On. Secrets: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY` (واختيارياً `TELEGRAM_BOT_TOKEN` و`TELEGRAM_CHAT_ID` للتقرير).

### 1) سحب كود شام من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 2) تحميل آخر نقطة لشام مع أداة تقسيم النص وأداتي الصورة والصوت المحفوظة معها

In [ ]:
import torch
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset
from tokenizer_select import select_pretrained_tokenizer
from train_image_tokenizer import save_tokenizer_checkpoint as save_image_tokenizer
from train_audio_tokenizer import save_tokenizer_checkpoint as save_audio_tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")

CHAT_OWN = fetch_dataset("sham-chat-checkpoint")
_ck = sorted(CHAT_OWN.rglob("final_chat.pt")) if CHAT_OWN else []
if _ck:
    base_ckpt, BASE_DIR = _ck[0], _ck[0].parent
    print("استئناف مرحلة المحادثة من ناتجها السابق.")
else:
    CHAT_OWN = None
    _mm = fetch_dataset("sham-multimodal-checkpoint")
    _ck = sorted(_mm.rglob("final_" + "multimodal.pt")) if _mm else []
    assert _ck, "لم يُنشر ناتج المرحلة الثانية (sham-multimodal-checkpoint) بعد — شغّلي دفتر المرحلة الثانية أولاً."
    base_ckpt, BASE_DIR = _ck[0], _ck[0].parent
    print("بداية مرحلة المحادثة من ناتج المرحلة الثانية.")

model, start_step, _ = load_checkpoint(base_ckpt, map_location=device)
tokenizer = ShamTextTokenizer.load(str(sorted(BASE_DIR.rglob("sham_small_tokenizer.json"))[0]))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"نقطة الانطلاق: {base_ckpt.name} — الخطوة {start_step:,}، vocab={tokenizer.vocab_size:,}")

# نفس أداتي الترميز اللتين تدرّب بهما النموذج (المحفوظتين مع النقطة نفسها).
_img, _aud = select_pretrained_tokenizer("image", BASE_DIR), select_pretrained_tokenizer("audio", BASE_DIR)
assert _img and _aud, f"أداتا الصورة/الصوت غير موجودتين بجانب {base_ckpt.name} — أعيدي تشغيل المرحلة الثانية لتنشرهما."
(image_tokenizer, _img_step, _), (audio_tokenizer, _aud_step, _) = _img, _aud
image_tokenizer, audio_tokenizer = image_tokenizer.to("cpu").eval(), audio_tokenizer.to("cpu").eval()
save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_img_step)
save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_aud_step)
print("✅ أداتا الصورة والصوت من نفس النقطة.")

### 3) المقياس الثابت — قبل التدريب

In [ ]:
from sham_eval import evaluate, fixed_eval_pairs, format_eval

EVAL_PAIRS = fixed_eval_pairs()
model.to(device)
eval_before = evaluate(model, tokenizer, device, chat=CHAT_OWN is not None, pairs=EVAL_PAIRS)
print(format_eval("قبل الجلسة", start_step, eval_before))

### 4) بيانات هذه الجلسة: حوار جديد + وسائط داخل المحادثة + نص للتذكّر

In [ ]:
import json, random
from sham_chat import (load_arabic_dialogues, load_progress, build_chat_example, build_media_chat_examples,
                       batch_examples, text_replay_batches, interleave)
from sham_data_sources import Ledger, collect
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

MAX_DIALOGUES = 4_000      # أمثلة حوار جديدة في كل جلسة
MAX_MEDIA = 600            # صور + مقاطع صوت للمحادثة المتعددة الوسائط (من كل نوع)
CHAT_BATCH = 8
MAX_LEN = min(512, model.cfg.max_seq_len)

progress = load_progress([CHAT_OWN]) if CHAT_OWN else {}
dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
if not dialogues:  # انتهت كل الأمثلة المتاحة: دورة جديدة عليها (مع ما تعلّمه من وسائط ونص بينهما)
    progress = {"epoch": int(progress.get("epoch", 0)) + 1}
    dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
print(f"أمثلة حوار جديدة: {len(dialogues):,} (الدورة {progress.get('epoch', 0)})")
chat_examples = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in dialogues]

rng = random.Random(start_step)
media_examples = []
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
for kind in ("image", "audio"):
    ledger = Ledger.load(kind, name=f"chat_{kind}")
    manifest, _stats = collect(kind, f"/kaggle/working/corpus/{kind}", MAX_MEDIA, ledger, image_size=image_tokenizer.cfg.image_size)
    ledger.save("/kaggle/working/checkpoints")
    if kind == "image":
        ds = ImageCaptionDataset(manifest, image_size=image_tokenizer.cfg.image_size)
        encode = MultimodalCollator(tokenizer, image_tokenizer).encode_batch_images
    else:
        ds = AudioTranscriptDataset(manifest, n_mels=audio_tokenizer.cfg.n_mels, segment_frames=audio_tokenizer.cfg.segment_frames)
        encode = AudioMultimodalCollator(tokenizer, audio_tokenizer).encode_batch_audio
    for s in range(0, len(ds), 16):
        items = [ds[i] for i in range(s, min(s + 16, len(ds)))]
        with torch.no_grad():
            ids = encode(items)
        for (caption, _), media in zip(items, ids):
            media_examples += build_media_chat_examples(tokenizer.encode, caption, media.tolist(), kind, rng)
print(f"أمثلة وسائط داخل المحادثة: {len(media_examples):,}")

chat_batches = batch_examples(chat_examples, CHAT_BATCH, seed=start_step)
media_batches = batch_examples(media_examples, 4, seed=start_step + 1)
text_batches = text_replay_batches(tokenizer, MAX_LEN, 4, max(len(chat_batches) // 2, 1), "/kaggle/working/replay_text",
                                   max_position=20_000, seed=start_step)
all_batches = interleave([chat_batches, media_batches, text_batches], [1.0, 0.6, 0.5], seed=start_step)
print(f"دفعات: حوار {len(chat_batches):,} + وسائط {len(media_batches):,} + تذكّر نص {len(text_batches):,} = {len(all_batches):,}")
assert all_batches, "لا توجد بيانات لهذه الجلسة."

### 5) التدريب (على أي معالج تم اختياره، ضمن وقت الجلسة)

In [ ]:
import time
from train import TrainConfig, train, build_optimizer

GRAD_ACCUM = 2
MAX_TRAINING_HOURS = 3.5 if device.startswith("cuda") else 8.5
optimizer = build_optimizer(model, lr=5e-5, weight_decay=0.1)

t0 = time.time()
model.train()
for _b in all_batches[:5]:  # قياس السرعة الحقيقية على هذا الجهاز (لا تحديث للأوزان)
    ids, labels = _b if isinstance(_b, tuple) else (_b, _b)
    _, loss = model(ids.to(device), labels=labels.to(device))
    loss.backward()
optimizer.zero_grad()
per_batch = max((time.time() - t0) / min(5, len(all_batches)), 1e-3)
budget = int(MAX_TRAINING_HOURS * 3600 * 0.85 / per_batch)
session_batches = all_batches[:budget]
TOTAL_STEPS = max(len(session_batches) // GRAD_ACCUM, 1)
print(f"~{per_batch:.2f} ث/دفعة على {device} — {len(session_batches):,} دفعة = {TOTAL_STEPS:,} خطوة تحديث.")

train_cfg = TrainConfig(
    seq_len=MAX_LEN, batch_size=CHAT_BATCH, grad_accum_steps=GRAD_ACCUM, lr=5e-5,
    warmup_steps=max(10, TOTAL_STEPS // 50), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=200, log_every=20,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)
loss_history = train(model, session_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=optimizer)
final_step = start_step + len(loss_history)
print(f"انتهى: {len(loss_history):,} خطوة — الخطوة النهائية {final_step:,}")

### 6) المقياس الثابت — بعد التدريب، وتقرير تيليجرام

In [ ]:
eval_after = evaluate(model, tokenizer, device, chat=True, pairs=EVAL_PAIRS)
report = format_eval("مرحلة المحادثة", final_step, eval_after, before=eval_before)
print(report)
try:
    from kaggle_secrets import UserSecretsClient
    _tg_token = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    _tg_chat = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    _tg_token, _tg_chat = None, None
from telegram_report import send_telegram_message
send_telegram_message(_tg_token, _tg_chat, report)

### 7) الحفظ والنشر إلى `sham-chat-checkpoint`

In [ ]:
import shutil
from checkpoint import save_checkpoint
from sham_inputs import publish_dataset

save_checkpoint("/kaggle/working/checkpoints/final_chat.pt", model, final_step)
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
shutil.copy2("/kaggle/working/checkpoints/final_chat.pt", upload_dir / "final_chat.pt")
for _f in ("sham_small_tokenizer.json", "image_tokenizer.pt", "audio_tokenizer.pt"):
    shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    shutil.copy2(_ledger, upload_dir / _ledger.name)
(upload_dir / "chat_progress.json").write_text(json.dumps(progress), encoding="utf-8")
(upload_dir / "eval_history.json").write_text(json.dumps(
    {"step": final_step, "before": {k: v for k, v in eval_before.items() if k != "samples"},
     "after": {k: v for k, v in eval_after.items() if k != "samples"}}, ensure_ascii=False), encoding="utf-8")
publish_dataset(upload_dir, "sham-chat-checkpoint", f"chat stage at step {final_step:,}")